In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-7: Whose samples? Off-policy vs. on-policy

## RFT Alternative methods, part 2 of the RLLM series


## Where this session fits

> **RLLM-6's limit: PPO and GRPO use only the model's own samples (GRPO gives one number per completion, PPO one per token from a critic). Why look at other methods? They differ on three axes (RLLM-6). Here: axis 1.**

- **Part 2: RFT Alternative methods** (RFT: reinforcement fine-tuning, RLLM-1): alternatives to PPO and GRPO
- **Headline of Part 2**: off-policy versus on-policy, axis 1. What you hold (checker, teacher, budget) decides what it allows (RLLM-13)
- **Position**: session 7 of 14, after part 1 (RLLM-1 to RLLM-6, ending with GRPO and its refinements)
- **Goal of Part 2**: a **map**: three axes place any method
- **Roadmap**: 7 whose samples (here), 8 weights, 9 to 11 four methods, 12 the teacher, 13 the map, 14 the checks
  - the four: 9 self-distillation and RAFT, 10 DPO, 11 on-policy distillation (OPD)

<img src="images/series_map_7.png" width=55%>


## This session: RLLM-7

- **Question**: **whose samples** does an update use (axis 1), and what does that cost?
- **By the end you can**:
  - define **off-policy** and **on-policy**, and place SFT, distillation, PPO and GRPO
  - say how **stale** (written by an older version of the model) a batch can be and still count as nearly on-policy
  - say which **prefix** (the prompt plus the output tokens so far) an update conditions on (imposed from outside, or the model's own)
  - name what each of off-policy and on-policy buys and costs


## Map: this session's two choices, and the three axes

> **Methods are placed by three axes. This session covers axis 1, whose samples, and a separate choice: which prefix.**

<img src="images/rllm7_map.png" alt="the three axes with axis 1 highlighted; an empty grid of whose samples by which prefix; the sections that cover each part" width=54%>

- This is the **plan**: the grid fills in the **Prefix** section
- The **prefix** is the prompt plus the output tokens so far
- The **Dial** section sets the rows, **Prefix** the columns, **Buys and costs** the two diagonal cells (one slide each)

## Map: the three axes, as a table

| Axis | The question | The answers |
|---|---|---|
| **1. Whose samples** (*this session*) | whose completions does the update use? | another model's or a dataset's, or the model's own |
| **2. Who supplies the weights** (RLLM-8) | something to **imitate** or to **maximize**? | a **reference** (a dataset's token, or a teacher's distribution per token, or a number per token read from it), or a **reward** (a score, or a **preference pair**: a better and a worse completion of one prompt) |
| **3. How fine** (RLLM-8) | how many numbers do the weights carry? | one per completion, one per sampled token, or a distribution per token |

- A **weight** is the multiplier an update puts on a token (RLLM-8)
- The network's numbers are **parameters**
- The **prefix** is a separate choice, **not a fourth axis**: the three axes alone place a method
- A **placement**, not a ranking: each method fits some situations
- Next, the **Dial** section: whose samples, from own to others'


## Dial: off-policy and on-policy, not a switch

> **On-policy: the update uses completions the current model just wrote. Off-policy: it uses completions from anywhere else. Between them: a dial of staleness.**

<img src="images/staleness_dial.png" width=62%>

- **Own** samples: written by the current model, or a copy within a few updates
- **Others'**: stored data, other models, older checkpoints
- **Stale**: written by an earlier version than the one now updated (RLLM-4: a batch reused for several updates)
- It describes **the data**, not the loss
- **PPO** reuses a batch: the ratio $\pi_\theta / \pi_{\text{old}}$ corrects the drift and the clip limits it (RLLM-4)


## Dial: the words, and where you met the distinction

> **We use RL to fine-tune a language model: it writes, something rates or corrects what it wrote, and an update changes its parameters.**

- **Prompt** $\x$, **completion** $\y$ (the whole output, one *episode*), **policy** $\pi_\theta$ (the model)
- **Sample**: $\y \sim \pi_\theta(\cdot \mid \x)$. **Update**: one gradient step on a batch
- **Reference**: what an update asks the model to reproduce, such as a dataset's answer or a teacher's distribution (axis 2, RLLM-8)

| Algorithm | Samples the update uses | Kind |
|---|---|---|
| **Q-learning, DQN** (replay buffer) | transitions from any behavior, often old ones | off-policy |
| **SARSA, REINFORCE** | the current policy's own | on-policy |
| **PPO, GRPO** | a slightly older copy of the policy, corrected by the ratio | nearly on-policy |
| **SFT on a dataset** | someone else's completions, never the model's own | the extreme off-policy case |

- The **reference model** of RLLM-4 (the frozen KL anchor) is a different thing from a **reference**
- Next: how stale a batch can be


## Dial: how stale is too stale? The clip fraction

> **At step 8 the ratio is 1.32, outside the clip range. The clip then bites on one side only: above 1.2 for a positive advantage, below 0.8 for a negative one.**

*Synthetic numbers.* One token, sampled by the old model with
probability $0.25$. Its **ratio** is the current probability over the old
one:

| Step on the batch | Current probability | Ratio | Inside the clip range $[0.8, 1.2]$? |
|---|---|---|---|
| 0 (fresh) | 0.25 | 1.00 | yes |
| 4 | 0.28 | 1.12 | yes |
| 8 | 0.33 | 1.32 | **no** |

- With a **positive advantage**, this token gets no further gradient. With a **negative** one the clip bites only when the ratio falls below $0.8$
- **The check:** the **share of tokens whose ratio has left the clip range** (the *clip fraction*), or the average log-ratio
  - the clip range $\epsilon = 0.2$ is a typical PPO setting (RLLM-4)
- **Our judgment:** *on-policy enough* while most ratios sit near $1$
- The dial sets whose samples; the **Prefix** section adds a second, separate choice


A loop with no PPO ratio can compute the same ratio from stored
log-probabilities of the generating model.


## Prefix: which prefix does the update condition on?

> **The prefix is the prompt plus the output tokens so far: the RL *state* of generation. Teacher forcing: the prefix is the reference's tokens, not the model's own.**

<img src="images/prefix_choice.png" width=66%>

- **Teacher forcing**: the prefix is the *reference's* tokens, whatever the model would have written
  - (the **reference**, defined in the Dial section, is whatever supplies the tokens: a dataset or a model, not necessarily a stronger one)
- **Free generation**: the prefix is the model's *own* previous tokens
- Whose samples (rows) and which prefix (columns) are separate choices: the next slide crosses them


Teacher forcing is about the *prefix*, not about the loss function:

- Scoring a sequence's log-probability is **one parallel forward pass**,
  whether the sequence came from a dataset or from your own sampler
- The code is the same either way
- What makes it *teacher forcing* is that the prefix being conditioned on
  is **the reference's**, not the model's own
- Careful: some code and papers call *any* parallel scoring pass
  "teacher-forced". In this series we do not; we use the narrower,
  reference-prefix meaning


## Prefix: the two choices crossed

> **Whose samples (rows) and which prefix (columns) are separate choices; one combination carries no information by itself.**

*This is the map's empty grid, filled in. It is a different grid from RLLM-8's map (whose samples x who supplies the weights).*

| | **A prefix imposed from outside** (teacher forcing) | **The model's own generated tokens** (no prefix imposed) |
|---|---|---|
| **Another model's or a dataset's samples** (off-policy) | SFT on a dataset, classical distillation (a teacher's saved outputs), another model's samples kept if a checker accepts (RLLM-9's hybrid applied to another model's samples; RLLM-9 itself uses the model's own) | *(empty in this series: another model's completions always carry that model's prefixes; re-weighting by a ratio corrects the samples, not the prefix)* |
| **The model's own samples** (on-policy) | **degenerate** (forward reference to RLLM-8, which explains it): the model's own tokens as the reference carry no information, no push on average, **unless a reward picks which completions to keep** | PPO, GRPO, OPD (on-policy distillation, RLLM-11) |


## Prefix: reading the grid

- With no reward, teacher forcing needs a reference from **elsewhere**
- RLLM-9 uses the degenerate cell: a reward picks which of the model's own completions to keep
- OPD (on-policy distillation, RLLM-11) is a different cell: the model's own prefix, with a teacher's reference
- **Costed next: the two diagonal cells.** The off-diagonal ones have nothing of their own to cost
  - own samples with an imposed prefix is degenerate without a reward (above)
  - others' samples with the model's own prefix is empty (above)
- **Own**: the current model or a copy within a few updates. **Others'**: stored data, other models, older checkpoints

> **Keep this: teacher forcing means the prefix is the reference's. It returns in RLLM-11, where the teacher reads the student's prefix.**
<!-- seed-plant: teacher-forcing-prefix -->

- Next: what the rows and columns each buy and cost


## Buys and costs: others' samples with an imposed prefix

<img src="images/state_coverage.png" width=50%>

- A **state** is a prefix; the **support** is the region the training data covers (dashed)
- **Buys**: no generation in the loop; stable labels
  - a fixed dataset, reused for many epochs
  - it can teach what the model would never produce itself
- **Costs**: **exposure bias** (contested): the model trains only on the dataset's prefixes but generates from its own
  - a mistake of its own makes a prefix it was rarely shown, so it may not practise recovery
- **Capped by the data**, and no signal about which of the model's own *sampled* outputs are bad


## Buys and costs: on-policy samples and the model's own prefix

<img src="images/reach_limit.png" width=55%>

- Completions A to H are candidate answers, ordered from most to least likely under the model
- **Buys**: trains on the prefixes you will actually visit, so train and
  test conditioning match
  - it sees its own mistakes, and an external signal can flag them
  - it generates its own raw material
- **Costs**: you can only *directly* learn from what you sample (a
  **reach limit**)
  - generation sits inside the loop, so sampling is slow
  - your own samples say nothing about what is good: a rater, a
    verifier or a teacher must supply that


## Hand-off: where this returns

> **Whose samples, and which prefix, is the first question every later method answers.**

| Session | Where it meets this |
|---|---|
| **RLLM-8** | the map of axes: own samples with no reward push nowhere on average |
| **RLLM-9** sample, filter, imitate | the model's own samples, filtered by a reward, then many steps on one batch: the staleness dial |
| **RLLM-10** DPO | pairs written earlier: off-policy, nearer on-policy if resampled |
| **RLLM-11** on-policy distillation | the student's own samples and prefixes, read by a teacher: own samples, no prefix imposed |
| **RLLM-13** the map and decision tool | every method placed by whose samples it uses |


## Think about

1. A model is fine-tuned to near-zero loss on a clean dataset and still
   produces bad completions when it generates freely. Why?
2. A method trains on another model's saved completions but keeps only
   those a checker accepts. Off-policy or on-policy? Which column of the
   grid is it in? What can it **not** teach?
3. Name something an on-policy method *cannot* teach, however long you
   train. What would you add?
4. A loop takes eight gradient steps on one fresh batch. How on-policy
   is step eight, and what would you check (hint: the worked number)?


## What we established, and what it sets up

- **Established**: **off-policy** means samples from elsewhere and **on-policy** the model's own, with a dial between. The prefix is a second, separate choice
- **Limit of this session**: it says **where samples come from**, not **what the update does with them**
- **Sets up**: RLLM-8, Who supplies the weights? A reference or a reward
  - who supplies the weights: a **reference** to imitate or a **reward** to maximize (axis 2), and how fine (axis 3)?


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Kevin Lu et al. (Thinking Machines Lab), *On-Policy Distillation*, blog post, Oct 2025](https://thinkingmachines.ai/blog/on-policy-distillation/): the opening section (samples, signal)
  - [Lilian Weng, *Policy Gradient Algorithms*, Lil'Log, 2018](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/): the policy-gradient basics and the off-policy section, a refresher on what "on-policy" means
- **Additional** (4 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved


## Aside: two words Part 2 avoids

- Part 2 never says "score" (the Intro's score function) or "target" (its TD target) for what an update consumes
- We say **reference** and **reward** instead


## Aside: is exposure bias where hallucination comes from?

<img src="images/hallucination_causes.png" width=75%>

- **Partly.** Exposure bias is one plausible explanation of why an error
  *persists and compounds* ("snowballing"); it is itself contested
- It does not by itself explain where the first error *starts*
- How much each cause contributes is an open empirical question


Notes for the aside above. Two places to read further:

- **Zhang et al. (2023)**, *How Language Model Hallucinations Can
  Snowball*: errors compound once a model has committed to a wrong
  prefix, and models can often recognize those errors as wrong when
  asked separately, so a knowledge gap is not the whole story
- **Gekhman et al. (2024)**, *Does Fine-Tuning LLMs on New Knowledge
  Encourage Hallucinations?*: fine-tuning on facts the model does not
  already know can increase hallucination
- Treat each as evidence for **one mechanism**, not as a full account
- The left box of the figure is our reading, not a result from these papers
